# 07 — Pruning vs. Scratch

## Reset vs revision under budget constraints

This notebook turns the central comparison from arXiv:2606.14150 into
a small repo artifact:

- **scratch training** → reset
- **pruning + retraining** → revision
- **matched budget comparison** → observe

It creates:

```text
results/07_pruning_vs_scratch.csv
figures/07_pruning_vs_scratch.png
```

No model downloads. No GPU. No pruning jobs.

In [ ]:
from pathlib import Path
import sys

# Works from repo root or from notebooks/
cwd = Path.cwd().resolve()
repo = cwd.parent if cwd.name == "notebooks" else cwd
src = repo / "src"

if str(src) not in sys.path:
    sys.path.insert(0, str(src))

print("repo:", repo)
print("src :", src)
print("src exists:", src.exists())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from pruning_rml.comparisons import label_mode

# Prefer the existing repo helper.
# Fall back locally only if an older paths.py is missing ensure_dirs.
try:
    from pruning_rml.paths import ensure_dirs
except ImportError:
    def ensure_dirs(root, names=("results", "figures", "reports")):
        root = Path(root)
        outputs = {}
        for name in names:
            path = root / name
            path.mkdir(parents=True, exist_ok=True)
            outputs[name] = path
        return outputs

outputs = ensure_dirs(repo)
results = outputs["results"]
figures = outputs["figures"]

print("results:", results)
print("figures:", figures)

## Comparison schema

This notebook is intentionally qualitative.

The paper's exact benchmark values can be added later. For now, the goal is
to create a clean table that captures the experimental distinction:

```text
reset
    =
train small model from random initialization

revision
    =
start from larger pretrained model, prune, retrain, evaluate
```

In [ ]:
rows = [
    {
        "method": "training from scratch",
        "mode": label_mode("training from scratch"),
        "starting_point": "random initialization",
        "training_budget": "matched tokens",
        "expected_behavior": "must rebuild capability",
        "rml_interpretation": "reset tests reconstruction",
        "revision_score": 1,
    },
    {
        "method": "pruned + retrained",
        "mode": label_mode("pruned + retrained"),
        "starting_point": "larger pretrained model",
        "training_budget": "matched tokens",
        "expected_behavior": "inherits useful structure",
        "rml_interpretation": "revision tests surviving residues",
        "revision_score": 3,
    },
    {
        "method": "scratch + full pipeline budget",
        "mode": label_mode("training from scratch"),
        "starting_point": "random initialization",
        "training_budget": "larger token budget",
        "expected_behavior": "can become competitive",
        "rml_interpretation": "reset can catch up with enough compute",
        "revision_score": 2,
    },
    {
        "method": "fine-grained pruning",
        "mode": label_mode("fine-grained pruning"),
        "starting_point": "larger pretrained model",
        "training_budget": "matched or continued budget",
        "expected_behavior": "may retain advantage longer",
        "rml_interpretation": "granular revision preserves more structure",
        "revision_score": 4,
    },
]

df = pd.DataFrame(rows)
df

In [ ]:
csv_path = results / "07_pruning_vs_scratch.csv"
df.to_csv(csv_path, index=False)

print("saved:", csv_path)
print("exists:", csv_path.exists())

## Visual summary

The plot below is not a benchmark plot. It is a repo-orientation figure:

- lower score means less inherited structure
- higher score means more revision / inheritance / residue preservation

Later notebooks can replace this with paper-derived metrics.

In [ ]:
fig_path = figures / "07_pruning_vs_scratch.png"

plot_df = df[["method", "revision_score"]].copy()

ax = plot_df.plot.bar(
    x="method",
    y="revision_score",
    legend=False,
    figsize=(9, 5),
)

ax.set_title("Pruning vs. scratch: reset and revision")
ax.set_xlabel("Method")
ax.set_ylabel("Revision / inheritance score")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(fig_path, dpi=160)
plt.show()

print("saved:", fig_path)
print("exists:", fig_path.exists())

In [ ]:
# Optional: inspect created files

for folder in ["results", "figures"]:
    path = repo / folder
    print(folder, "exists:", path.exists())
    if path.exists():
        print(" ", [p.name for p in path.iterdir()])

## Notebook result

Notebook 07 creates the first measured repo artifacts:

```text
results/07_pruning_vs_scratch.csv
figures/07_pruning_vs_scratch.png
```

Next notebook:

```text
notebooks/13_granularity.ipynb
```

Next question:

> Which pruning granularities preserve the most useful residues?